# 🧠 Aula 06a: Transição para Deep Learning Moderno com PyTorch
### Da Implementação Manual em NumPy aos Frameworks de Diferenciação Automática

---

## 🎯 Objetivo Pedagógico
Nas aulas anteriores (01 a 05), construímos passo a passo toda a fundação matemática e algorítmica das Redes Neurais:
- Perceptron clássico e regra de convergência;
- Adaline, gradiente descendente estocástico (SGD) e derivadas de funções de custo;
- Espaços latentes e camadas ocultas para superar a barreira linear (XOR);
- Retropropagação de erros (*Backpropagation*) e regra da cadeia multivariada.

Nesta aula, daremos o passo definitivo para o **Deep Learning profissional**:
1. **Por que frameworks modernos?**: Em modelos profundos com milhões de parâmetros, calcular derivadas parciais e matrizes jacobianas manualmente no papel se torna inviável. Entra em cena o **Autograd** (Diferenciação Automática em Grafos Computacionais);
2. **Integração PyTorch + Scikit-Learn**: Empacotaremos modelos PyTorch (`nn.Sequential`, `nn.Linear`, `nn.ReLU`, `optim.Adam`) dentro de classes compatíveis com a API padrão do `scikit-learn` (`BaseEstimator`, `ClassifierMixin`), mantendo o familiar fluxo `fit` e `predict`;
3. **Progressão Arquitetural**: Compararemos na prática:
   - **SLP (Single Layer Perceptron)**: Classificação linear direta dos dígitos;
   - **SHLP (Single Hidden Layer Perceptron)**: Rede com 1 camada oculta e ReLU;
   - **MLP Profundo Genérico**: Rede dinâmica com N camadas ocultas configuráveis;
4. **Auditoria e Contagem de Parâmetros**: Desvendaremos como calcular e auditar com precisão o número de pesos ($W$) e biases ($b$) que uma rede neural aprende na memória.

```
   Fluxo de Trabalho com PyTorch:
   [Entrada X] ---> [nn.Linear] ---> [nn.ReLU] ---> [nn.Linear] ---> [Predição]
                                                                          |
                                                                    [Função de Custo]
                                                                          |
   [Update de Pesos] <--- [optimizer.step()] <--- [loss.backward()] <--- [Loss (Erro)]
```

## 1. Carregamento e Preparação do Dataset Digits

Para demonstrar o poder das redes neurais, utilizaremos o benchmark clássico de visão computacional **Digits** (embutido no Scikit-Learn):
- **Instâncias**: Imagens em escala de cinza de dígitos numéricos manuscritos ($0$ a $9$);
- **Resolução**: Cada imagem tem $8 \times 8$ pixels, resultando em um vetor de entrada de $64$ características (`features`);
- **Divisão Treino/Teste**: Separamos $80\%$ para treino e $20\%$ para teste com semente fixa (`random_state=42`) para garantir reprodutibilidade.

In [2]:
# Importa o conjunto de dados clássico de dígitos manuscritos (8x8 pixels)
from sklearn.datasets import load_digits
# Importa a função utilitária para particionar os dados entre treino e teste
from sklearn.model_selection import train_test_split

# Carrega a matriz de atributos X (imagens vetorizadas de 64 pixels) e os rótulos y (dígitos de 0 a 9)
X, y = load_digits(return_X_y=True)
# Divide o dataset: 80% para aprendizado (treino) e 20% para validação imparcial (teste)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Exibe a dimensionalidade dos conjuntos de dados gerados
print(f"Conjunto de Treino: {X_train.shape[0]} amostras, {X_train.shape[1]} pixels/features por imagem")
print(f"Conjunto de Teste:  {X_test.shape[0]} amostras")
print(f"Classes a classificar: {sorted(list(set(y_train)))}")

## 2. SLPClassifier: Single Layer Perceptron com PyTorch

Nesta primeira implementação, conectamos os $64$ pixels diretamente aos $10$ neurônios de saída, sem nenhuma camada intermediária:
- **`nn.Linear(64, 10)`**: Realiza a transformação afim $Z = X W^T + b$;
- **`nn.Softmax(dim=1)`**: Converte os logits de cada classe em probabilidades normalizadas ($\sum p_i = 1$);
- **`nn.CrossEntropyLoss()`**: Função de perda para classificação multiclasse;
- **`optim.Adam`**: Otimizador com momento adaptativo para convergência rápida;
- **Ciclo canônico do PyTorch**:
  1. `optimizer.zero_grad()`: Zera os gradientes acumulados da iteração anterior;
  2. `outputs = self.model(X)`: Realiza a propagação direta (*forward pass*);
  3. `loss = criterion(outputs, y)`: Avalia a discrepância entre predições e rótulos reais;
  4. `loss.backward()`: Calcula as derivadas parciais via autodiferenciação (*backward pass*);
  5. `optimizer.step()`: Atualiza os pesos e biases no sentido do gradiente descendente.

> [!NOTE]
> **Integração com Scikit-Learn**: Ao herdar de `BaseEstimator` e `ClassifierMixin`, nossa classe se torna perfeitamente compatível com o ecossistema Scikit-Learn (podendo ser usada em `cross_val_score`, `GridSearchCV` e `Pipeline`).

In [18]:
# Importa as classes base do Scikit-Learn para criação de estimadores customizados
from sklearn.base import BaseEstimator, ClassifierMixin
# Importa a métrica de acurácia para avaliação do desempenho
from sklearn.metrics import accuracy_score
# Importa o NumPy para manipulação numérica e conversão de tensores
import numpy as np
# Importa o ecossistema PyTorch e seus submódulos neurais e de otimização
import torch
import torch.nn as nn
import torch.optim as optim

# Define o classificador Perceptron de Camada Única (SLP) compatível com Scikit-Learn
class SLPClassifier(BaseEstimator, ClassifierMixin):
  def __init__(self, max_iter=200, learning_rate=0.01):
    # Define o número máximo de épocas de treinamento
    self.max_iter = max_iter
    # Define a taxa de aprendizado para o otimizador Adam
    self.learning_rate = learning_rate
    # Inicializa o atributo do modelo neural como nulo até a execução do fit
    self.model = None

  def fit(self, X, y):
    # Mapeia as classes únicas e obtém seus índices ordinais inteiros consecutivos
    self.labels, ids = np.unique(y, return_inverse=True)
    # Extrai o número de variáveis preditoras de entrada (64 pixels)
    input_shape = X.shape[1]
    # Extrai o número de neurônios da camada de saída (10 classes)
    output_shape = len(self.labels)

    # Converte os dados NumPy para tensores PyTorch adequados (float32 para entradas, long para índices de classe)
    X = torch.tensor(X, dtype=torch.float32)
    y = torch.tensor(ids, dtype=torch.long)

    # Monta o grafo computacional sequencial: transformação linear seguida de softmax
    self.model = nn.Sequential(
        nn.Linear(input_shape, output_shape),
        nn.Softmax(dim=1)
    )
    # Define a função de custo Entropia Cruzada Multiclasse
    criterion = nn.CrossEntropyLoss()
    # Instancia o otimizador Adam para atualizar os parâmetros treináveis da rede
    optimizer = optim.Adam(self.model.parameters(), lr=self.learning_rate)

    # Loop de treinamento por épocas sucessivas
    for epoch in range(self.max_iter):
      # 1. Zera os gradientes acumulados nos tensores de parâmetros
      optimizer.zero_grad()
      # 2. Forward pass: alimenta o modelo e gera as probabilidades
      outputs = self.model(X)
      # 3. Calcula o custo da predição atual em relação aos alvos reais
      loss = criterion(outputs, y)
      # 4. Backward pass: calcula gradientes via autodiferenciação
      loss.backward()
      # 5. Otimização: aplica a atualização dos pesos e biases
      optimizer.step()

    return self

  def predict_proba(self, X):
    # Converte a matriz de entrada em tensor PyTorch
    X = torch.tensor(X, dtype=torch.float32)
    # Desabilita o cálculo de gradientes durante a inferência para economizar memória e processamento
    with torch.no_grad():
      outputs = self.model(X)
    # Retorna o resultado convertido de volta para array NumPy
    return outputs.numpy()

  def predict(self, X):
    # Obtém as probabilidades estimadas para cada classe
    proba = self.predict_proba(X)
    # Identifica o índice da classe de maior probabilidade prevista (Argmax)
    idx_pred = np.argmax(proba, axis=1)
    # Decodifica o índice de volta para o rótulo original do dataset
    return self.labels[idx_pred]

# Instancia o classificador linear com 10.000 iterações e taxa de aprendizado 0.0001
model = SLPClassifier(10000, 0.0001)
# Treina o modelo nos dígitos de treinamento
model.fit(X_train, y_train)
# Realiza a predição no conjunto de teste independente
y_pred = model.predict(X_test)
# Calcula e imprime a acurácia global alcançada pelo modelo linear
print("Acurácia do SLP (Linear) no teste:", accuracy_score(y_test, y_pred))

0.9611111111111111


## 3. SHLPClassifier: Inserindo uma Camada Oculta com ReLU

Como vimos na aula do XOR, modelos puramente lineares não conseguem modelar fronteiras complexas e não-lineares.
No `SHLPClassifier` (Single Hidden Layer Perceptron):
1. Adicionamos uma **camada oculta** intermediária com $100$ neurônios (`nn.Linear(input_shape, 100)`);
2. Aplicamos a função de ativação **ReLU** ($f(z) = \max(0, z)$), que introduz não-linearidade suave e evita o problema do desaparecimento de gradiente;
3. Conectamos os $100$ neurônios ocultos à camada de saída de $10$ neurônios (`nn.Linear(100, output_shape)`).

```
   [64 pixels] ---> [100 Neurônios Ocultos] ---> [ReLU] ---> [10 Classes]
```

In [21]:
# Importa as classes base e utilitários necessários
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.metrics import accuracy_score
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

# Define o classificador com 1 Camada Oculta (SHLP)
class SHLPClassifier(BaseEstimator, ClassifierMixin):
  def __init__(self, n_hidden=100, max_iter=200, learning_rate=0.01):
    # Define a quantidade de neurônios na camada oculta intermediária
    self.n_hidden = n_hidden
    # Define o total de épocas de treinamento
    self.max_iter = max_iter
    # Define a taxa de aprendizado
    self.learning_rate = learning_rate
    self.model = None

  def fit(self, X, y):
    # Mapeia as classes exclusivas do target
    self.labels, ids = np.unique(y, return_inverse=True)
    input_shape = X.shape[1]      # 64 entradas
    output_shape = len(self.labels) # 10 saídas

    # Conversão dos dados para tensores de precisão compatível com PyTorch
    X = torch.tensor(X, dtype=torch.float32)
    y = torch.tensor(ids, dtype=torch.long)

    # Arquitetura com camada oculta não-linear: Linear -> ReLU -> Linear -> Softmax
    self.model = nn.Sequential(
        nn.Linear(input_shape, self.n_hidden), # Projeção de 64 para 100 dimensões
        nn.ReLU(),                            # Ativação não-linear retificadora
        nn.Linear(self.n_hidden, output_shape), # Mapeamento de 100 para 10 classes
        nn.Softmax(dim=1)                     # Normalização em probabilidades
    )
    # Custo e otimizador
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(self.model.parameters(), lr=self.learning_rate)

    # Treinamento via gradiente estocástico com otimizador Adam
    for epoch in range(self.max_iter):
      optimizer.zero_grad()
      outputs = self.model(X)
      loss = criterion(outputs, y)
      loss.backward()
      optimizer.step()

    return self

  def predict_proba(self, X):
    X = torch.tensor(X, dtype=torch.float32)
    with torch.no_grad():
      outputs = self.model(X)
    return outputs.numpy()

  def predict(self, X):
    proba = self.predict_proba(X)
    idx_pred = np.argmax(proba, axis=1)
    return self.labels[idx_pred]

# Instancia a rede com 1 camada oculta de 100 neurônios e treina por 10.000 iterações
model = SHLPClassifier(100, 10000, 0.0001)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("Acurácia do SHLP (1 Camada Oculta) no teste:", accuracy_score(y_test, y_pred))

0.9861111111111112


## 4. MLPClassfier: Rede Neural Profunda com Múltiplas Camadas Dinâmicas

Para tornar o modelo verdadeiramente escalável e profundo, o professor implementou o `MLPClassfier`:
- Ele aceita uma **lista de inteiros** no parâmetro `n_hidden` (ex: `[100]`, `[50, 50]`, `[128, 64, 32]`);
- Constrói a lista de camadas dinamicamente em um laço `for` encadeando `nn.Linear` e `nn.ReLU`;
- Usa o desempacotamento de lista `nn.Sequential(*layers)` para registrar toda a arquitetura no PyTorch com elegância e sem código repetido.

In [24]:
# Importa as classes base necessárias para compatibilidade com o Scikit-Learn
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.metrics import accuracy_score
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

# Define o classificador MLP com número arbitrário de camadas ocultas
class MLPClassfier(BaseEstimator, ClassifierMixin):
  def __init__(self, n_hidden=[100], max_iter=200, learning_rate=0.01):
    # Lista contendo o número de neurônios de cada camada oculta consecutiva
    self.n_hidden = n_hidden
    self.max_iter = max_iter
    self.learning_rate = learning_rate
    self.model = None

  def fit(self, X, y):
    # Mapeia as classes distintas
    self.labels, ids = np.unique(y, return_inverse=True)
    input_shape = X.shape[1]
    output_shape = len(self.labels)

    # Converte os arrays para tensores PyTorch
    X = torch.tensor(X, dtype=torch.float32)
    y = torch.tensor(ids, dtype=torch.long)

    # Constrói dinamicamente a sequência de camadas neurais
    # 1. Primeira camada oculta: conecta a entrada (input_shape) ao primeiro bloco oculto
    layers = [nn.Linear(input_shape, self.n_hidden[0]), nn.ReLU()]
    # 2. Camadas ocultas intermediárias: conecta a camada (i-1) à camada (i)
    for i in range(1, len(self.n_hidden)):
      layers.append(nn.Linear(self.n_hidden[i-1], self.n_hidden[i]))
      layers.append(nn.ReLU())
    # 3. Camada de saída: conecta a última camada oculta ao número de classes
    layers.append(nn.Linear(self.n_hidden[-1], output_shape))
    # 4. Camada de ativação final: probabilidades via Softmax
    layers.append(nn.Softmax(dim=1))
    # Empacota todas as camadas desempacotadas dentro de um container Sequencial
    self.model = nn.Sequential(*layers)

    # Custo e otimizador
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(self.model.parameters(), lr=self.learning_rate)

    # Ciclo de treinamento por épocas
    for epoch in range(self.max_iter):
      optimizer.zero_grad()
      outputs = self.model(X)
      loss = criterion(outputs, y)
      loss.backward()
      optimizer.step()

    return self

  def predict_proba(self, X):
    X = torch.tensor(X, dtype=torch.float32)
    with torch.no_grad():
      outputs = self.model(X)
    return outputs.numpy()

  def predict(self, X):
    proba = self.predict_proba(X)
    idx_pred = np.argmax(proba, axis=1)
    return self.labels[idx_pred]

# Instancia o modelo MLP profundo com 1 camada oculta de 100 neurônios
model = MLPClassfier([100], 10000, 0.0001)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("Acurácia do MLP ([100]) no teste:", accuracy_score(y_test, y_pred))

0.9833333333333333


## 5. Auditoria de Parâmetros: PyTorch vs Cálculo Teórico

Quantos parâmetros livres (graus de liberdade) uma rede neural precisa ajustar na memória?
Em uma camada totalmente conectada (`Linear`), cada neurônio possui um **peso para cada entrada** mais um **termo de bias** independente:
$$\text{Parâmetros} = (\text{Entradas} \times \text{Neurônios}) + \text{Biases}$$

Para a rede `[100]`:
- Camada de Entrada para Oculta: $64 \times 100 = 6.400$ pesos $+ 100$ biases $= 6.500$;
- Camada Oculta para Saída: $100 \times 10 = 1.000$ pesos $+ 10$ biases $= 1.010$;
- **Total Teórico com Biases**: $6.500 + 1.010 = 7.510$ parâmetros;
- **Apenas Pesos (sem biases)**: $6.400 + 1.000 = 7.400$.

In [25]:
# Calcula o número total real de parâmetros treináveis gerenciados pelo PyTorch (pesos + biases)
total_params = sum(p.numel() for p in model.model.parameters())
print(f"Total de parâmetros registrados no PyTorch (pesos + biases): {total_params}")

Total number of parameters in model.model: 7510


In [31]:
# Contabiliza teoricamente apenas os pesos multiplicativos entre camadas (desconsiderando os biases)
n_feats = X_train.shape[1]        # 64 atributos de entrada
n_ihw = n_feats * 100             # Pesos da camada de entrada para a camada oculta (64 * 100 = 6400)
n_how = 100 * len(set(y_train))   # Pesos da camada oculta para a camada de saída (100 * 10 = 1000)
print(f"Número teórico de pesos puros (sem bias): {n_ihw + n_how}")
print(f"Diferença exata referente aos biases (100 + 10): {total_params - (n_ihw + n_how)}")

Number of input hidden weights: 7400


## 6. Avaliando um MLP Mais Profundo: 2 Camadas Ocultas `[50, 50]`

O que acontece quando aumentamos a profundidade da rede para $2$ camadas ocultas de $50$ neurônios cada?
- Entrada: $64$ pixels;
- Oculta 1: $50$ neurônios ($64 \times 50 = 3.200$ pesos);
- Oculta 2: $50$ neurônios ($50 \times 50 = 2.500$ pesos);
- Saída: $10$ classes ($50 \times 10 = 500$ pesos);
- Total de pesos: $3.200 + 2.500 + 500 = 6.200$;
- Biases: $50 + 50 + 10 = 110$;
- **Total Geral**: $6.200 + 110 = 6.310$ parâmetros!

In [32]:
# Treina o MLP com duas camadas ocultas sucessivas de 50 neurônios cada ([50, 50])
model = MLPClassfier([50, 50], 10000, 0.0001)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
# Avalia o desempenho no conjunto de teste
print("Acurácia do MLP ([50, 50]) no teste:", accuracy_score(y_test, y_pred))

0.9777777777777777


In [33]:
# Mede o total de parâmetros registrados pelo PyTorch no modelo com 2 camadas ocultas
total_params = sum(p.numel() for p in model.model.parameters())
print(f"Total real de parâmetros no modelo PyTorch ([50, 50]): {total_params}")

Total number of parameters in model.model: 6310


In [34]:
# Cálculo analítico detalhado da arquitetura de 2 camadas ocultas
n_feats = X_train.shape[1]        # 64 entradas
n_ihw = n_feats * 50              # Entrada -> Oculta 1: 64 * 50 = 3200
n_hhw = 50 * 50                   # Oculta 1 -> Oculta 2: 50 * 50 = 2500
n_how = 50 * len(set(y_train))   # Oculta 2 -> Saída: 50 * 10 = 500
n_pesos = n_ihw + n_hhw + n_how   # Soma dos pesos = 6200
n_biases = 50 + 50 + len(set(y_train)) # Soma dos biases = 110

print(f"Número de pesos de conexão (sem bias): {n_pesos}")
print(f"Número total teórico com biases (6200 + 110): {n_pesos + n_biases}")
print(f"Validação: Coincide exatamente com o PyTorch? {total_params == (n_pesos + n_biases)}")

Number of input hidden weights: 6200


## 7. 🎓 Conclusão e Resumo do Curso de Redes Neurais Artificiais (ANN)

Parabéns! Com este caderno, você concluiu a jornada completa de Redes Neurais:

| Etapa | Abordagem | Conceitos Centrais |
| :--- | :--- | :--- |
| **Aulas 01-02** | Fundamentos Lineares | Perceptron, Adaline, Solução Fechada (Equação Normal), SGD, Bias e Vetorização |
| **Aulas 03-04** | O Desafio da Não-Linearidade | Funções de Custo, Multiclasse, Crise do XOR, Espaço Latente, Tanh e ELM |
| **Aula 05** | MLP Manual do Zero | Backpropagation analítico, Jacobiana, Regra da Cadeia e Camadas Arbitrárias em NumPy |
| **Aula 06** | Deep Learning Moderno | Grafos Dinâmicos, Autograd no PyTorch, Otimizador Adam, ReLU e Auditoria de Parâmetros |

> [!TIP]
> **Próximos Passos no Estudo**: Com a base matemática sólida desenvolvida ao longo destes 21 cadernos, você está plenamente capacitado para avançar para redes convolucionais (CNNs para visão computacional), redes recorrentes/LSTMs e modelos baseados em Transformers (Atenção e LLMs) utilizando o ecossistema PyTorch!